In [10]:
import pandas as pd
import numpy as np
from EssSimulation_withoutMaxDemand import EssSimulationModel
import calendar
import copy
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif']=['SimHei']    # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号

In [11]:
exp_name = "estimate1016"
node_name = "route_A"

In [12]:
es_info = {"transform_capacity": 630000,
           "invertband": 0,
           "soc_redundant_ratio": 0,
            "usable_depth": 0.95,
            "charge_loss": 0.92,
            "discharge_loss": 0.95,
            "es_charge_max": 12500,
            "es_charge_min": -12500,
            "es_capacity_max": 25000,
            "es_capacity_min": 0}
max_demand_price = 37

In [13]:
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/schedule_result_no_exceed_dod95.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

In [14]:
simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0) #4050
origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, max_demand_price)

In [15]:
origin_balance - opt_balance

5982658.443665415

In [16]:
5982658.44 + 6012485.58

11995144.02

In [17]:
es_charge_df['value'][es_charge_df['value'] > 0].sum() / 4

18760216.935529

In [18]:
es_charge_df.to_csv(f"./data/{exp_name}/{node_name}/opt_result/simulation_result_no_exceed_dod95.csv")